In [2]:
import os
import time
import hashlib
import pandas as pd
from selenium.webdriver.common.by import By
import undetected_chromedriver as uc

In [2]:
# Leer el token desde el archivo externo
ruta_token = "auth.tk"
try:
    with open(ruta_token, "r", encoding="utf-8") as archivo:
        mi_auth_token = archivo.read().strip()
except FileNotFoundError:
    print(f"ERROR: No se encontró el archivo '{ruta_token}'.")
    print("Crea el archivo en la misma carpeta del script, pega tu token dentro y vuelve a ejecutar.")
    driver.quit()
    exit()

In [ ]:
def seudonimizar_usuario(username):
    """Aplica un hash SHA-256 al nombre de usuario para cumplir con la Ley 21.719."""
    if not username: return None
    return hashlib.sha256(username.encode('utf-8')).hexdigest()

def extraer_comentarios_x(url, periodo, driver, limite=300):
    driver.get(url)
    print(f"Cargando {periodo}...")
    time.sleep(6) # Pausa inicial para carga de DOM
    
    comentarios_extraidos = []
    ids_vistos = set()
    intentos_sin_nuevos = 0 
    botones_cliqueados = set() # <--- Memoria para no repetir clics en el mismo botón
    
    while len(comentarios_extraidos) < limite:
        
        # Salvaguarda de navegación
        url_base = url.split('?')[0]
        if not driver.current_url.startswith(url_base):
            print("El navegador se desvió de la página principal. Regresando...")
            driver.get(url)
            time.sleep(5)
            
        cantidad_anterior = len(comentarios_extraidos)
        articulos = driver.find_elements(By.XPATH, '//article[@data-testid="tweet"]')
        
        for articulo in articulos:
            try:
                # 1. Obtener enlace único del tweet
                link_elem = articulo.find_elements(By.XPATH, './/a[contains(@href, "/status/")]')
                tweet_id = link_elem[0].get_attribute('href') if link_elem else None
                
                if not tweet_id or tweet_id in ids_vistos:
                    continue
                
                # 2. Extraer texto del tweet
                texto_elem = articulo.find_elements(By.XPATH, './/div[@data-testid="tweetText"]')
                texto = texto_elem[0].text if texto_elem else None
                
                # 3. Detectar si contiene multimedia
                media_elem = articulo.find_elements(By.XPATH, './/div[@data-testid="tweetPhoto"] | .//div[@data-testid="videoPlayer"]')
                tiene_media = len(media_elem) > 0
                
                if not texto:
                    if tiene_media:
                        texto = "[Contenido multimedia sin texto]"
                    else:
                        continue 
                
                # 4. Extraer usuario y fecha
                user_elem = articulo.find_elements(By.XPATH, './/div[@dir="ltr"]/span')
                usuario_raw = user_elem[0].text if user_elem else "usuario_desconocido"
                
                time_elem = articulo.find_elements(By.XPATH, './/time')
                fecha_hora = time_elem[0].get_attribute('datetime') if time_elem else None
                
                ids_vistos.add(tweet_id)
                
                comentarios_extraidos.append({
                    "id_texto": len(ids_vistos),
                    "fecha_hora": fecha_hora,
                    "id_usuario_hash": seudonimizar_usuario(usuario_raw),
                    "texto": texto,
                    "consulta_origen": url,
                    "periodo": periodo
                })
                
                if len(comentarios_extraidos) >= limite:
                    break
                    
            except Exception:
                continue 
                
        if len(comentarios_extraidos) >= limite:
            break
            
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3) 
        
        # Control de scroll y botones de expansión / spam
        if len(comentarios_extraidos) == cantidad_anterior:
            intentos_sin_nuevos += 1
            
            try:
                botones = driver.find_elements(By.XPATH, "//*[contains(text(), 'Mostrar') or contains(text(), 'spam') or contains(text(), 'Show')]")
                for btn in botones:
                    try:
                        # Identificamos el botón por su contenido único
                        btn_id = btn.get_attribute('outerHTML')
                        if btn.is_displayed() and btn_id not in botones_cliqueados:
                            driver.execute_script("arguments[0].click();", btn)
                            botones_cliqueados.add(btn_id) # Registramos que ya fue presionado
                            print("Botón de 'Mostrar' o 'spam' presionado con éxito...")
                            time.sleep(3)
                            break
                    except Exception:
                        continue
            except Exception:
                pass
                
            # Si tras 5 intentos el scroll ya no trae comentarios nuevos, salimos limpiamente
            if intentos_sin_nuevos >= 5: 
                print(f"Fondo de la página alcanzado. Obtenidos: {len(comentarios_extraidos)} de {limite}.")
                break
        else:
            intentos_sin_nuevos = 0 
            print(f"Extrayendo... {len(comentarios_extraidos)} / {limite}")
            
    return comentarios_extraidos[:limite]
# ==========================================
# 1. Configurar el WebDriver Indetectable
# ==========================================
ruta_perfil_local = os.path.abspath(os.path.join(os.getcwd(), "perfil_selenium_uc"))
if not os.path.exists(ruta_perfil_local):
    os.makedirs(ruta_perfil_local)

opciones = uc.ChromeOptions()
opciones.add_argument(f"--user-data-dir={ruta_perfil_local}") 
opciones.add_argument("--disable-dev-shm-usage")

opciones.add_argument("--lang=en-US")
prefs = {"intl.accept_languages": "en,en_US"}
opciones.add_experimental_option("prefs", prefs)

# Iniciar el navegador. Si sigues con el error de versión, usa: uc.Chrome(options=opciones, version_main=155)
driver = uc.Chrome(options=opciones, version_main=155)

# ==========================================
# 2. Login por Inyección de Cookie
# ==========================================
print("Inyectando sesión de X...")
driver.get("https://x.com/robots.txt")
time.sleep(2)


driver.add_cookie({
    'name': 'auth_token',
    'value': mi_auth_token,
    'domain': '.x.com'
})

# Refrescar la página principal
driver.get("https://x.com/")
time.sleep(5)
print("Sesión inyectada con éxito. Iniciando extracción...")

# ==========================================
# 3. Recolección de Datos
# ==========================================
links = [
    {"url": "https://x.com/NintendoAmerica/status/2064361089668620543?s=20", "periodo": "Antes (Expectativa / Anuncio)"},
    {"url": "https://x.com/NintendoAmerica/status/2097333400054927619?s=20", "periodo": "Después (Tráiler Gameplay)"}
]

datos_totales = []
for link in links:
    datos = extraer_comentarios_x(link["url"], link["periodo"], driver, limite=500)
    datos_totales.extend(datos)

driver.quit()

# ==========================================
# 4. Estructurar en Pandas y exportar
# ==========================================
df_comentarios = pd.DataFrame(datos_totales)

# Prevenir error si la muestra solicitada es mayor a los datos obtenidos
muestra_size = min(50, len(df_comentarios))
df_muestra_ruido = df_comentarios.sample(n=muestra_size, random_state=42) if muestra_size > 0 else pd.DataFrame()

df_comentarios.to_csv("corpus_ocarina_remake.csv", sep=';', index=False, encoding="utf-8")
if not df_muestra_ruido.empty:
    df_muestra_ruido.to_csv("muestra_ruido_manual.csv", sep=';', index=False, encoding="utf-8")

print("\nExtracción completada. Primeras 5 filas:")
print(df_comentarios.head())

Inyectando sesión de X...
Sesión inyectada con éxito. Iniciando extracción...
Cargando Antes (Expectativa / Anuncio)...
Extrayendo... 5 / 300
Extrayendo... 19 / 300
Extrayendo... 41 / 300
Extrayendo... 56 / 300
Extrayendo... 74 / 300
Extrayendo... 91 / 300
Extrayendo... 112 / 300
Extrayendo... 130 / 300
Extrayendo... 157 / 300
Extrayendo... 186 / 300
Extrayendo... 219 / 300
Extrayendo... 222 / 300
Botón de 'Mostrar' o 'spam' presionado con éxito...
Extrayendo... 231 / 300
Extrayendo... 232 / 300
Botón de 'Mostrar' o 'spam' presionado con éxito...
El navegador se desvió de la página principal. Regresando...
Fondo de la página alcanzado. Obtenidos: 232 de 300.
Cargando Después (Tráiler Gameplay)...
Extrayendo... 5 / 300
Extrayendo... 22 / 300
Extrayendo... 44 / 300
Extrayendo... 55 / 300
Extrayendo... 69 / 300
Extrayendo... 87 / 300
Extrayendo... 110 / 300
Extrayendo... 134 / 300
Extrayendo... 159 / 300
Extrayendo... 179 / 300
Extrayendo... 197 / 300
Extrayendo... 198 / 300
Botón de 'Mos

In [7]:
import pandas as pd

df_comentarios = pd.read_csv("corpus_ocarina_remake.csv", sep=';', encoding="utf-8")
df_comentarios.head()

,id_texto,fecha_hora,id_usuario_hash,texto,consulta_origen,periodo
0,1,2026-06-09T14:56:52.000Z,400b585eb60519835945bb0fa92adfd7977f7369241350...,The Legend of Zelda: Ocarina of Time will be r...,https://x.com/NintendoAmerica/status/206436108...,Antes (Expectativa / Anuncio)
1,2,2026-06-09T14:57:32.000Z,961d77778edf61e276b5a7952e551462685566d82021ac...,"""Hey can we see Ocarina of Time gameplay?""\n\n...",https://x.com/NintendoAmerica/status/206436108...,Antes (Expectativa / Anuncio)
2,3,2026-06-09T15:01:03.000Z,31ed9acf9453843e09587a7e6ed5b05b3d7e48921da4b9...,We're getting him in HD we are not ready for this,https://x.com/NintendoAmerica/status/206436108...,Antes (Expectativa / Anuncio)
3,4,2026-06-09T16:19:01.000Z,90a0d8e9c0416f10d8f02429c3af3c3f49a5f8a98278fc...,,https://x.com/NintendoAmerica/status/206436108...,Antes (Expectativa / Anuncio)
4,5,2026-06-09T16:17:01.000Z,4b6c621736bc9eaac24e6c4c82cd858fd60bebe4f12668...,My honest reaction to this information.,https://x.com/NintendoAmerica/status/206436108...,Antes (Expectativa / Anuncio)


Web scrapping Youtube

In [4]:
import os
import time
import hashlib
import pandas as pd
from selenium.webdriver.common.by import By
import undetected_chromedriver as uc

def seudonimizar_usuario(username):
    """Aplica un hash SHA-256 al nombre de usuario para cumplir con la Ley 21.719."""
    if not username: return None
    return hashlib.sha256(username.encode('utf-8')).hexdigest()

def extraer_comentarios_youtube(url, periodo, driver, limite=500):
    driver.get(url)
    print(f"Cargando video ({periodo})...")
    time.sleep(5) # Pausa inicial para carga de la página
    
    # En YouTube, para que los comentarios carguen, debemos hacer un pequeño scroll inicial hacia abajo
    print("Desplazando hacia la sección de comentarios...")
    driver.execute_script("window.scrollTo(0, 500);")
    time.sleep(3)
    
    comentarios_extraidos = []
    ids_vistos = set()
    intentos_sin_nuevos = 0 
    
    # Bucle de scroll para cargar comentarios dinámicos de YouTube
    while len(comentarios_extraidos) < limite:
        cantidad_anterior = len(comentarios_extraidos)
        
        # Localizar los contenedores de los comentarios en YouTube
        comentarios_elements = driver.find_elements(By.CSS_SELECTOR, 'ytd-comment-thread-renderer')
        
        for elem in comentarios_elements:
            try:
                # 1. Extraer el texto del comentario principal
                texto_elem = elem.find_elements(By.CSS_SELECTOR, '#content-text')
                texto = texto_elem[0].text if texto_elem else None
                
                # 2. Extraer el nombre de usuario
                user_elem = elem.find_elements(By.CSS_SELECTOR, '#author-text span')
                usuario_raw = user_elem[0].text.strip() if user_elem else "usuario_desconocido"
                
                # 3. Extraer la fecha o tiempo relativo (ej. "Hace 2 días")
                time_elem = elem.find_elements(By.CSS_SELECTOR, '#header-author yt-formatted-string.published-time-text a')
                fecha_hora = time_elem[0].text if time_elem else None
                
                # ID único combinando usuario y texto truncado (o ID del elemento si está disponible)
                comentario_id = f"{usuario_raw}-{fecha_hora}-{text_hash(text=texto)}"
                
                if texto and comentario_id not in ids_vistos:
                    ids_vistos.add(comentario_id)
                    
                    comentarios_extraidos.append({
                        "id_texto": len(ids_vistos),
                        "fecha_relativa": fecha_hora,
                        "id_usuario_hash": seudonimizar_usuario(usuario_raw),
                        "texto": texto,
                        "consulta_origen": url,
                        "periodo": periodo
                    })
                    
                    if len(comentarios_extraidos) >= limite:
                        break
            except Exception:
                continue 
                
        if len(comentarios_extraidos) >= limite:
            break
            
        # Hacer scroll hacia abajo para activar la carga de más comentarios
        driver.execute_script("window.scrollBy(0, 1000);")
        time.sleep(3) 
        
        # Validación de control para evitar bucles infinitos si llegamos al final
        if len(comentarios_extraidos) == cantidad_anterior:
            intentos_sin_nuevos += 1
            if intentos_sin_nuevos >= 5:
                print(f"Fondo de los comentarios alcanzado. Obtenidos: {len(comentarios_extraidos)} de {limite}.")
                break
        else:
            intentos_sin_nuevos = 0 
            print(f"Extrayendo YouTube... {len(comentarios_extraidos)} / {limite}")
            
    return comentarios_extraidos[:limite]

def text_hash(text):
    """Función auxiliar para crear un identificador único seguro basado en el contenido."""
    if not text: return ""
    return hashlib.md5(text.encode('utf-8')).hexdigest()[:8]

# ==========================================
# 1. Configurar el WebDriver Indetectable
# ==========================================
ruta_perfil_local = os.path.abspath(os.path.join(os.getcwd(), "perfil_selenium_yt"))
if not os.path.exists(ruta_perfil_local):
    os.makedirs(ruta_perfil_local)

opciones = uc.ChromeOptions()
opciones.add_argument(f"--user-data-dir={ruta_perfil_local}") 
opciones.add_argument("--disable-dev-shm-usage")
# Forzar idioma en inglés si deseas evitar traducciones automáticas de YouTube
opciones.add_argument("--lang=en-US")

driver = uc.Chrome(options=opciones, version_main=155)

# ==========================================
# 2. Recolección de Datos (Ejemplo con dos videos)
# ==========================================

links_youtube = [
    {"url": "https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS", "periodo": "Antes (Anuncio)"},
    {"url": "https://youtu.be/wuFfiTEr2yc?si=gzKDvpmqZWdGgInH", "periodo": "Después (Gameplay)"}
]

datos_totales = []
for link in links_youtube:
    datos = extraer_comentarios_youtube(link["url"], link["periodo"], driver, limite=500)
    datos_totales.extend(datos)

driver.quit()

# ==========================================
# 3. Estructurar en Pandas y exportar
# ==========================================
df_comentarios = pd.DataFrame(datos_totales)

muestra_size = min(50, len(df_comentarios))
df_muestra_ruido = df_comentarios.sample(n=muestra_size, random_state=42) if muestra_size > 0 else pd.DataFrame()

df_comentarios.to_csv("corpus_youtube_ocarina.csv", sep=';', index=False, encoding="utf-8")
if not df_muestra_ruido.empty:
    df_muestra_ruido.to_csv("muestra_ruido_youtube.csv", sep=';', index=False, encoding="utf-8")

print("\n¡Extracción de YouTube completada con éxito!")
print(df_comentarios.head())

Cargando video (Antes (Anuncio))...
Desplazando hacia la sección de comentarios...
Extrayendo YouTube... 20 / 500
Extrayendo YouTube... 40 / 500
Extrayendo YouTube... 60 / 500
Extrayendo YouTube... 80 / 500
Extrayendo YouTube... 100 / 500
Extrayendo YouTube... 120 / 500
Extrayendo YouTube... 140 / 500
Extrayendo YouTube... 160 / 500
Extrayendo YouTube... 180 / 500
Extrayendo YouTube... 199 / 500
Extrayendo YouTube... 219 / 500
Extrayendo YouTube... 239 / 500
Extrayendo YouTube... 259 / 500
Extrayendo YouTube... 279 / 500
Extrayendo YouTube... 299 / 500
Extrayendo YouTube... 319 / 500
Extrayendo YouTube... 339 / 500
Extrayendo YouTube... 359 / 500
Extrayendo YouTube... 379 / 500
Extrayendo YouTube... 399 / 500
Extrayendo YouTube... 419 / 500
Extrayendo YouTube... 439 / 500
Extrayendo YouTube... 459 / 500
Extrayendo YouTube... 479 / 500
Extrayendo YouTube... 499 / 500
Cargando video (Después (Gameplay))...
Desplazando hacia la sección de comentarios...
Extrayendo YouTube... 20 / 500
Extr

In [ ]:
import pandas as pd

df_comentarios = pd.read_csv("corpus_youtube_ocarina.csv", sep=';', encoding="utf-8")
df_comentarios.head() 

## Preprocesamiento del corpus (comentarios mayoritariamente en ingl?s)

Se conserva `texto` sin modificar. La rama de sentimiento mantiene negaciones, emojis, puntuaci?n y expresiones originales; solo normaliza Unicode/espacios y reemplaza URLs y menciones por marcadores. La rama de temas pasa a min?sculas, elimina s?mbolos tipo emoji y filtra una lista peque?a de stopwords inglesas. No se traduce el texto: los comentarios mixtos conservan sus palabras. Revisen las salidas antes de usar estos datos.

In [ ]:
import html
import re
import unicodedata

import pandas as pd

if "texto" not in df_comentarios.columns:
    raise ValueError("El corpus debe tener una columna llamada 'texto'.")

STOPWORDS_TEMAS_EN = {
    "a", "an", "and", "are", "as", "at", "be", "been", "but", "by",
    "for", "from", "had", "has", "have", "he", "her", "hers", "him",
    "his", "i", "if", "in", "into", "is", "it", "its", "me", "my",
    "of", "on", "or", "our", "ours", "she", "so", "that", "the",
    "their", "them", "there", "they", "this", "to", "us", "was",
    "we", "were", "what", "when", "which", "who", "with", "you", "your"
}

def normalizar_base(texto):
    """Normaliza Unicode y espacios sin cambiar el contenido ling??stico."""
    if pd.isna(texto):
        return ""
    texto = html.unescape(unicodedata.normalize("NFC", str(texto)))
    return re.sub(r"\s+", " ", texto).strip()

def preparar_para_sentimiento(texto):
    """Conserva negaciones, emojis, may?sculas y puntuaci?n expresiva."""
    texto = normalizar_base(texto)
    texto = re.sub(r"https?://\S+|www\.\S+", " URL ", texto, flags=re.IGNORECASE)
    texto = re.sub(r"(?<!\w)@[\w_]+", " USUARIO ", texto)
    texto = re.sub(r"(?<!\w)#([\w]+)", r"\1", texto)
    return re.sub(r"\s+", " ", texto).strip()

def quitar_simbolos_emoji(texto):
    """Quita s?mbolos emoji comunes para una variante de an?lisis de temas."""
    caracteres = []
    for caracter in texto:
        categoria = unicodedata.category(caracter)
        es_selector_o_union = caracter in {"\ufe0e", "\ufe0f", "\u200d"}
        if categoria not in {"So", "Sk"} and not es_selector_o_union:
            caracteres.append(caracter)
    return "".join(caracteres)

def preparar_para_temas(texto):
    """Crea tokens en min?scula y elimina stopwords inglesas frecuentes."""
    texto = quitar_simbolos_emoji(preparar_para_sentimiento(texto)).casefold()
    tokens = re.findall(r"[\w]+(?:['?][\w]+)?", texto, flags=re.UNICODE)
    tokens = [token for token in tokens if token not in STOPWORDS_TEMAS_EN]
    return " ".join(tokens)

df_comentarios["texto_sentimiento"] = df_comentarios["texto"].apply(preparar_para_sentimiento)
df_comentarios["texto_temas"] = df_comentarios["texto"].apply(preparar_para_temas)
df_comentarios["texto_sentimiento_sin_emojis"] = df_comentarios["texto_sentimiento"].apply(quitar_simbolos_emoji)

print(f"Comentarios procesados: {len(df_comentarios)}")
print("Columnas originales preservadas:", "texto" in df_comentarios.columns)

### Revisi?n manual de ejemplos
Comparen cinco textos originales con las dos versiones procesadas para detectar si alguna regla altera el significado.

In [ ]:
columnas_ejemplo = ["texto", "texto_sentimiento", "texto_temas"]
columnas_ejemplo = [col for col in columnas_ejemplo if col in df_comentarios.columns]
df_comentarios[columnas_ejemplo].head(5)

### Ablaci?n: conservar o quitar emojis
Esta comparaci?n mide cu?nto cambia el texto de entrada al eliminar emojis. Para cumplir la ablaci?n de sentimiento, apliquen ambos textos al mismo clasificador y conjunto anotado, manteniendo todo lo dem?s constante, y reporten F1-macro/matriz de confusi?n. Esta celda no inventa m?tricas de sentimiento: solo cuantifica el cambio de preprocesamiento.

In [ ]:
comparacion_emojis = pd.DataFrame({
    "con_emojis": df_comentarios["texto_sentimiento"],
    "sin_emojis": df_comentarios["texto_sentimiento_sin_emojis"]
})
comparacion_emojis["cambio"] = comparacion_emojis["con_emojis"] != comparacion_emojis["sin_emojis"]
comparacion_emojis["tokens_con_emojis"] = comparacion_emojis["con_emojis"].str.split().str.len()
comparacion_emojis["tokens_sin_emojis"] = comparacion_emojis["sin_emojis"].str.split().str.len()

resumen_ablacion_emojis = pd.DataFrame({
    "medida": ["comentarios", "comentarios cuyo texto cambia", "% de comentarios que cambia", "promedio de tokens con emojis", "promedio de tokens sin emojis"],
    "valor": [
        len(comparacion_emojis),
        int(comparacion_emojis["cambio"].sum()),
        round(100 * comparacion_emojis["cambio"].mean(), 2) if len(comparacion_emojis) else 0.0,
        round(comparacion_emojis["tokens_con_emojis"].mean(), 2),
        round(comparacion_emojis["tokens_sin_emojis"].mean(), 2)
    ]
})
resumen_ablacion_emojis